In [1]:
import os
import json
from datasets import load_dataset


def convert_split(split, name, out_dir):
    records = []

    kept = 0
    skipped_missing_fields = 0
    skipped_empty_answer = 0

    for i, ex in enumerate(split):
        question = ex.get("question")
        sparql_dbpedia = ex.get("sparql_dbpedia18")
        sparql_wikidata = ex.get("sparql_wikidata")
        solved_answer = ex.get("solved_answer")

        # Require required fields
        if not question or not sparql_dbpedia or not sparql_wikidata:
            skipped_missing_fields += 1
            continue

        # Require a non-empty result set
        if not solved_answer:
            skipped_empty_answer += 1
            continue

        records.append(
            {
                "id": i,
                "question": str(question).strip(),
                "sparql_dbpedia18": str(sparql_dbpedia).strip(),
                "sparql_wikidata": str(sparql_wikidata).strip(),
                "answer": solved_answer,
            }
        )
        kept += 1

    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, f"{name}.json")
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2)

    print(
        f"[{name}] kept={kept} | "
        f"skipped_missing_fields={skipped_missing_fields} | "
        f"skipped_empty_answer={skipped_empty_answer}"
    )
    print(f"Saved: {out_path}")


def main():
    print("Loading Orange/lc_quad2-sparqltotext ...")
    ds = load_dataset("Orange/lc_quad2-sparqltotext")
    print(ds)

    out_dir = "dataset"
    convert_split(ds["train"], "train", out_dir)
    convert_split(ds["validation"], "validation", out_dir)
    convert_split(ds["test"], "test", out_dir)


if __name__ == "__main__":
    main()

/upb/users/g/ganeshk/profiles/unix/cs/.conda/envs/sparql-emb/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading Orange/lc_quad2-sparqltotext ...
DatasetDict({
    train: Dataset({
        features: ['uid', 'NNQT_question', 'paraphrased_question', 'question', 'simplified_query', 'sparql_dbpedia18', 'sparql_wikidata', 'answer', 'solved_answer', 'subgraph', 'template', 'template_id', 'template_index'],
        num_rows: 21101
    })
    validation: Dataset({
        features: ['uid', 'NNQT_question', 'paraphrased_question', 'question', 'simplified_query', 'sparql_dbpedia18', 'sparql_wikidata', 'answer', 'solved_answer', 'subgraph', 'template', 'template_id', 'template_index'],
        num_rows: 3010
    })
    test: Dataset({
        features: ['uid', 'NNQT_question', 'paraphrased_question', 'question', 'simplified_query', 'sparql_dbpedia18', 'sparql_wikidata', 'answer', 'solved_answer', 'subgraph', 'template', 'template_id', 'template_index'],
        num_rows: 6024
    })
})
[train] kept=10551 | skipped_missing_fields=0 | skipped_empty_answer=10550
Saved: dataset/train.json
[validation] k

In [1]:
# rq1_qwen_sparql2text_encoder_eval_and_finetune.py
# Fine-tune Qwen on SPARQL -> Natural Text, then use the fine-tuned model
# as an encoder for retrieval (SPARQL embeddings).

import os
import gc
import json
import random
import inspect
from typing import Any, Dict, List, Tuple

import numpy as np
import torch
from datasets import Dataset
from peft import (
    LoraConfig,
    PeftConfig,
    PeftModel,
    get_peft_model,
    prepare_model_for_kbit_training,
)
from tqdm.auto import tqdm

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
)


# ============================================================
# Config
# ============================================================

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
DATA_DIR = "dataset"

USE_FIELD = "sparql_wikidata"  # or "sparql_wikidata"
SPARQL_VARIANT = USE_FIELD.replace("sparql_", "")
OUTPUT_DIR = f"./qwen_sparql2text_encoder_rq1_{SPARQL_VARIANT}"

MAX_SEQ_LENGTH = 768

TRAIN_BATCH_SIZE = 2
EVAL_BATCH_SIZE = 2
GRAD_ACCUM_STEPS = 4
NUM_EPOCHS = 3
LEARNING_RATE = 2e-4
SEED = 42

EMBED_BATCH_SIZE = 8

ANSWER_JACCARD_THRESHOLD = 0.8
THRESHOLD_SWEEP = [0.5, 0.6, 0.7, 0.8]

TOPK = [1, 5, 10]

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


# ============================================================
# Utils
# ============================================================

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)


def cleanup_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def load_json(path: str) -> List[Dict]:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def save_json(obj: Any, path: str):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


def l2_normalize(x: torch.Tensor, eps: float = 1e-12) -> torch.Tensor:
    return x / x.norm(dim=-1, keepdim=True).clamp(min=eps)


def cosine_sim_matrix(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    a_norm = a / (np.linalg.norm(a, axis=1, keepdims=True) + 1e-12)
    b_norm = b / (np.linalg.norm(b, axis=1, keepdims=True) + 1e-12)
    return a_norm @ b_norm.T


# ============================================================
# Prompting
# ============================================================

def format_sparql_to_text_prompt(sparql: str) -> str:
    return (
        "You are an expert in knowledge graphs.\n\n"
        "### Instruction\n"
        "Convert the SPARQL query into a clear natural language question.\n\n"
        f"### SPARQL Query\n{sparql}\n\n"
        "### Natural Text\n"
    )


# ============================================================
# Data loading
# ============================================================

def safe_list(x: Any) -> List[str]:
    if x is None:
        return []
    if isinstance(x, list):
        return [str(v).strip() for v in x if str(v).strip()]
    s = str(x).strip()
    return [s] if s else []


def normalize_answer_item(x: str) -> str:
    return " ".join(str(x).strip().split())


def normalize_answer_list(answer: Any) -> List[str]:
    vals = safe_list(answer)
    return [normalize_answer_item(v) for v in vals if normalize_answer_item(v)]


def load_examples(path: str, use_field: str) -> List[Dict[str, Any]]:
    raw = load_json(path)
    items = []

    for i, ex in enumerate(raw):
        sparql = ex.get(use_field)
        question = ex.get("question")
        answers = normalize_answer_list(ex.get("answer"))

        if sparql is None or not str(sparql).strip():
            continue
        if question is None or not str(question).strip():
            continue
        if len(answers) == 0:
            continue

        items.append(
            {
                "id": ex.get("id", i),
                "sparql": str(sparql).strip(),
                "question": str(question).strip(),
                "answer": answers,
            }
        )

    return items


# ============================================================
# Fine-tuning data: SPARQL -> natural language
# ============================================================

def build_prompt_and_target(example: Dict[str, Any]) -> Tuple[str, str]:
    prompt = format_sparql_to_text_prompt(example["sparql"])
    target = example["question"]
    return prompt, target


def tokenize_supervised_example(
    example: Dict[str, Any],
    tokenizer,
    max_length: int,
) -> Dict[str, Any]:
    prompt, target = build_prompt_and_target(example)

    prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
    target_ids = tokenizer(
        target + tokenizer.eos_token,
        add_special_tokens=False
    )["input_ids"]

    input_ids_full = prompt_ids + target_ids
    input_ids = input_ids_full[:max_length]

    attention_mask = [1] * len(input_ids)

    labels_full = ([-100] * len(prompt_ids)) + target_ids
    labels = labels_full[:max_length]

    return {
        "id": example["id"],
        "sparql": example["sparql"],
        "question": example["question"],
        "answer": example["answer"],
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }


class SupervisedDataCollator:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        max_len = max(len(f["input_ids"]) for f in features)

        pad_token_id = self.tokenizer.pad_token_id
        if pad_token_id is None:
            pad_token_id = self.tokenizer.eos_token_id

        input_ids, attention_mask, labels = [], [], []

        for f in features:
            pad_len = max_len - len(f["input_ids"])

            input_ids.append(f["input_ids"] + [pad_token_id] * pad_len)
            attention_mask.append(f["attention_mask"] + [0] * pad_len)
            labels.append(f["labels"] + [-100] * pad_len)

        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
        }


def build_tokenized_train_val_datasets(tokenizer):
    train_examples = load_examples(os.path.join(DATA_DIR, "train.json"), USE_FIELD)
    val_examples = load_examples(os.path.join(DATA_DIR, "validation.json"), USE_FIELD)

    train_tok = [
        tokenize_supervised_example(ex, tokenizer, MAX_SEQ_LENGTH)
        for ex in train_examples
    ]
    val_tok = [
        tokenize_supervised_example(ex, tokenizer, MAX_SEQ_LENGTH)
        for ex in val_examples
    ]

    train_ds = Dataset.from_list(train_tok)
    val_ds = Dataset.from_list(val_tok)
    return train_ds, val_ds


def log_truncation_stats(examples: List[Dict[str, Any]], tokenizer, max_length: int, split_name: str):
    lengths = []
    for ex in examples:
        prompt, target = build_prompt_and_target(ex)
        full = prompt + target + tokenizer.eos_token
        ids = tokenizer(full, add_special_tokens=False)["input_ids"]
        lengths.append(len(ids))

    if not lengths:
        print(f"\n=== Tokenization Stats ({split_name}) ===")
        print("No examples found.")
        return

    truncated = sum(1 for l in lengths if l > max_length)

    print(f"\n=== Tokenization Stats ({split_name}) ===")
    print(f"Total examples         : {len(lengths)}")
    print(f"Min length             : {min(lengths)}")
    print(f"Max length             : {max(lengths)}")
    print(f"Mean length            : {sum(lengths) / len(lengths):.1f}")
    print(
        f"Truncated (>{max_length}) : "
        f"{truncated} ({100.0 * truncated / len(lengths):.1f}%)"
    )


# ============================================================
# TrainingArguments
# ============================================================

def build_training_args() -> TrainingArguments:
    bf16_supported = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    use_fp16 = torch.cuda.is_available() and not bf16_supported
    use_bf16 = bf16_supported

    kwargs = dict(
        output_dir=OUTPUT_DIR,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM_STEPS,
        num_train_epochs=NUM_EPOCHS,
        learning_rate=LEARNING_RATE,
        logging_steps=10,
        eval_steps=100,
        save_strategy="steps",
        save_steps=100,
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        fp16=use_fp16,
        bf16=use_bf16,
        report_to="none",
    )

    sig = inspect.signature(TrainingArguments.__init__)
    if "eval_strategy" in sig.parameters:
        kwargs["eval_strategy"] = "steps"
    elif "evaluation_strategy" in sig.parameters:
        kwargs["evaluation_strategy"] = "steps"

    if "optim" in sig.parameters:
        kwargs["optim"] = "paged_adamw_8bit"

    return TrainingArguments(**kwargs)


# ============================================================
# QwenSPARQLEncoder
# ============================================================

class QwenSPARQLEncoder:
    def __init__(self, lora_path: str, device: str = DEVICE):
        self.device = device

        peft_cfg = PeftConfig.from_pretrained(lora_path)
        self.base_model_name = peft_cfg.base_model_name_or_path

        try:
            self.tokenizer = AutoTokenizer.from_pretrained(lora_path)
        except Exception:
            self.tokenizer = AutoTokenizer.from_pretrained(self.base_model_name)

        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        if device == "cuda":
            bnb_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_use_double_quant=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch.float16,
            )
            base_model = AutoModelForCausalLM.from_pretrained(
                self.base_model_name,
                quantization_config=bnb_config,
                device_map="auto",
            )
        else:
            base_model = AutoModelForCausalLM.from_pretrained(
                self.base_model_name,
                torch_dtype=torch.float32,
            ).to(device)

        self.model = PeftModel.from_pretrained(base_model, lora_path, is_trainable=False)
        self.model.eval()

    def build_encoder_prompt(self, sparql: str) -> str:
        return format_sparql_to_text_prompt(sparql)

    @torch.no_grad()
    def encode_sparql(
        self,
        sparql_list: List[str],
        batch_size: int = 8,
        max_length: int = 768,
    ) -> np.ndarray:
        all_embeddings: List[torch.Tensor] = []
        model_device = next(self.model.parameters()).device

        for start in tqdm(range(0, len(sparql_list), batch_size), desc="Encoding SPARQL"):
            batch = sparql_list[start: start + batch_size]
            prompts = [self.build_encoder_prompt(s) for s in batch]

            toks = self.tokenizer(
                prompts,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=max_length,
            )
            toks = {k: v.to(model_device) for k, v in toks.items()}

            outputs = self.model(
                **toks,
                output_hidden_states=True,
                return_dict=True,
            )
            last_hidden = outputs.hidden_states[-1]  # (B, T, H)

            attn = toks["attention_mask"].unsqueeze(-1).float()
            pooled = (last_hidden * attn).sum(dim=1) / attn.sum(dim=1).clamp(min=1e-9)
            pooled = l2_normalize(pooled)

            all_embeddings.append(pooled.cpu())

        return torch.cat(all_embeddings, dim=0).numpy()


# ============================================================
# Ground truth + Jaccard semantic relevance
# ============================================================

def answer_jaccard(a: List[str], b: List[str]) -> float:
    sa, sb = set(a), set(b)
    if len(sa) == 0 or len(sb) == 0:
        return 0.0
    return len(sa.intersection(sb)) / len(sa.union(sb))


def build_ground_truth_by_answer_jaccard(
    examples: List[Dict[str, Any]],
    threshold: float = 0.8,
    exclude_self: bool = True,
):
    n = len(examples)
    jaccard_matrix = np.zeros((n, n), dtype=np.float32)

    for i in tqdm(range(n), desc=f"Building answer-Jaccard matrix @ threshold={threshold}"):
        ai = examples[i]["answer"]
        for j in range(n):
            if exclude_self and i == j:
                continue
            aj = examples[j]["answer"]
            jaccard_matrix[i, j] = answer_jaccard(ai, aj)

    gt: Dict[int, List[int]] = {}
    for i in range(n):
        positives = [j for j in range(n) if jaccard_matrix[i, j] >= threshold]
        if exclude_self:
            positives = [j for j in positives if j != i]
        gt[i] = positives

    return gt, jaccard_matrix


def threshold_sweep(examples: List[Dict[str, Any]], thresholds: List[float] = None):
    if thresholds is None:
        thresholds = [0.5, 0.6, 0.7, 0.8]

    print("\n=== Jaccard Threshold Sensitivity ===")
    print(f"{'Threshold':>10} | {'Valid Anchors':>14} | {'Coverage %':>10}")
    print("-" * 42)

    summary = {}
    for t in thresholds:
        gt_t, _ = build_ground_truth_by_answer_jaccard(
            examples,
            threshold=t,
            exclude_self=True,
        )
        valid = sum(1 for i in range(len(examples)) if len(gt_t.get(i, [])) > 0)
        pct = 100.0 * valid / max(len(examples), 1)
        print(f"{t:>10.1f} | {valid:>14} | {pct:>9.1f}%")
        summary[str(t)] = {
            "valid_anchors": valid,
            "coverage_percent": pct,
        }

    return summary


# ============================================================
# Retrieval metrics
# ============================================================

def mask_self_matches(sim_matrix: np.ndarray) -> np.ndarray:
    sim = sim_matrix.copy()
    n, m = sim.shape
    if n == m:
        diag = np.arange(n)
        sim[diag, diag] = -1e9
    return sim


def get_ranked_indices(sim_matrix: np.ndarray) -> np.ndarray:
    return np.argsort(-sim_matrix, axis=1)


def recall_at_k(ranked_indices: np.ndarray, gt: Dict[int, List[int]], ks: List[int]) -> Dict[str, float]:
    ks = sorted(ks)
    valid_anchors = [i for i in range(ranked_indices.shape[0]) if len(gt.get(i, [])) > 0]

    out: Dict[str, float] = {}
    for k in ks:
        hits = 0
        for i in valid_anchors:
            gt_set = set(gt[i])
            topk = ranked_indices[i, :k]
            if gt_set.intersection(topk):
                hits += 1
        out[f"Recall@{k}"] = hits / max(len(valid_anchors), 1)
    return out


def mrr_at_k(ranked_indices: np.ndarray, gt: Dict[int, List[int]], k: int) -> float:
    valid_anchors = [i for i in range(ranked_indices.shape[0]) if len(gt.get(i, [])) > 0]
    rr = []

    for i in valid_anchors:
        gt_set = set(gt[i])
        topk = ranked_indices[i, :k]
        reciprocal_rank = 0.0
        for rank, idx in enumerate(topk, start=1):
            if idx in gt_set:
                reciprocal_rank = 1.0 / rank
                break
        rr.append(reciprocal_rank)

    return float(np.mean(rr)) if rr else 0.0


def dcg_at_k(relevances: List[float], k: int) -> float:
    rel = np.asarray(relevances[:k], dtype=np.float32)
    if rel.size == 0:
        return 0.0
    discounts = 1.0 / np.log2(np.arange(2, rel.size + 2))
    return float(np.sum(rel * discounts))


def ndcg_at_k(
    ranked_indices: np.ndarray,
    relevance_matrix: np.ndarray,
    gt: Dict[int, List[int]],
    k: int,
) -> float:
    valid_anchors = [i for i in range(ranked_indices.shape[0]) if len(gt.get(i, [])) > 0]
    vals = []

    for i in valid_anchors:
        topk = ranked_indices[i, :k]
        rels = [float(relevance_matrix[i, j]) for j in topk]
        dcg = dcg_at_k(rels, k)

        ideal_rels = sorted(
            [float(relevance_matrix[i, j]) for j in range(relevance_matrix.shape[1]) if j != i],
            reverse=True,
        )
        idcg = dcg_at_k(ideal_rels, k)
        vals.append(0.0 if idcg == 0.0 else dcg / idcg)

    return float(np.mean(vals)) if vals else 0.0


def best_jaccard_at_k(
    ranked_indices: np.ndarray,
    jaccard_matrix: np.ndarray,
    gt: Dict[int, List[int]],
    k: int,
) -> float:
    valid_anchors = [i for i in range(ranked_indices.shape[0]) if len(gt.get(i, [])) > 0]
    vals = []

    for i in valid_anchors:
        topk = ranked_indices[i, :k]
        vals.append(float(np.max(jaccard_matrix[i, topk])) if len(topk) > 0 else 0.0)

    return float(np.mean(vals)) if vals else 0.0


def mean_jaccard_at_k(
    ranked_indices: np.ndarray,
    jaccard_matrix: np.ndarray,
    gt: Dict[int, List[int]],
    k: int,
) -> float:
    valid_anchors = [i for i in range(ranked_indices.shape[0]) if len(gt.get(i, [])) > 0]
    vals = []

    for i in valid_anchors:
        topk = ranked_indices[i, :k]
        vals.append(float(np.mean(jaccard_matrix[i, topk])) if len(topk) > 0 else 0.0)

    return float(np.mean(vals)) if vals else 0.0


def evaluate_ranking(
    sim_matrix: np.ndarray,
    gt: Dict[int, List[int]],
    jaccard_matrix: np.ndarray,
    ks: List[int],
):
    sim_matrix = mask_self_matches(sim_matrix)
    ranked = get_ranked_indices(sim_matrix)

    results: Dict[str, float] = {}
    results.update(recall_at_k(ranked, gt, ks))

    max_k = max(ks)
    results[f"MRR@{max_k}"] = mrr_at_k(ranked, gt, max_k)

    for k in ks:
        results[f"nDCG@{k}"] = ndcg_at_k(ranked, jaccard_matrix, gt, k)
        results[f"BestJaccard@{k}"] = best_jaccard_at_k(ranked, jaccard_matrix, gt, k)
        results[f"MeanJaccard@{k}"] = mean_jaccard_at_k(ranked, jaccard_matrix, gt, k)

    results["NumAnchorsWithPositives"] = int(
        sum(1 for i in range(sim_matrix.shape[0]) if len(gt.get(i, [])) > 0)
    )
    return results, ranked


def print_results(title: str, results: Dict[str, float]):
    print(f"\n=== {title} ===")
    for k, v in results.items():
        if k == "NumAnchorsWithPositives":
            print(f"{k}: {v}")
        else:
            print(f"{k}: {v:.4f}")


def print_retrieval_examples(
    title: str,
    ranked: np.ndarray,
    examples: List[Dict[str, Any]],
    jaccard_matrix: np.ndarray,
    num_examples: int = 5,
):
    print(f"\n===== Retrieval preview: {title} =====")
    shown = 0

    for i in range(len(examples)):
        topk = ranked[i, :5]
        if topk.size == 0:
            continue

        print("\n" + "=" * 80)
        print(f"ANCHOR #{i} | id={examples[i]['id']}")
        print(f"SPARQL:\n{examples[i]['sparql']}")
        print(f"GOLD QUESTION:\n{examples[i]['question']}")
        print(f"GOLD ANSWER: {examples[i]['answer']}")

        print("\nTop-5 retrieved SPARQL queries:")
        for rank, j in enumerate(topk, start=1):
            print(f"\n[Rank {rank}] corpus_idx={j}")
            print(f"Question: {examples[j]['question']}")
            print(f"SPARQL: {examples[j]['sparql'][:200]}...")
            print(f"Answer: {examples[j]['answer']}")
            print(f"Answer-Jaccard(anchor, retrieved) = {jaccard_matrix[i, j]:.4f}")

        print("\nSummary:")
        print(f"Top-1 Jaccard: {jaccard_matrix[i, topk[0]]:.4f}")
        print(f"Best Jaccard in Top-3: {np.max(jaccard_matrix[i, topk[:3]]):.4f}")
        print(f"Best Jaccard in Top-5: {np.max(jaccard_matrix[i, topk[:5]]):.4f}")

        shown += 1
        if shown >= num_examples:
            break


# ============================================================
# Finetune + RQ1
# ============================================================

def finetune_qwen_sparql2text():
    ensure_dir(OUTPUT_DIR)
    cleanup_memory()

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    train_raw = load_examples(os.path.join(DATA_DIR, "train.json"), USE_FIELD)
    val_raw = load_examples(os.path.join(DATA_DIR, "validation.json"), USE_FIELD)

    log_truncation_stats(train_raw, tokenizer, MAX_SEQ_LENGTH, "train")
    log_truncation_stats(val_raw, tokenizer, MAX_SEQ_LENGTH, "validation")

    train_ds, val_ds = build_tokenized_train_val_datasets(tokenizer)

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto",
    )

    model = prepare_model_for_kbit_training(model)
    model.config.use_cache = False

    if hasattr(model, "gradient_checkpointing_enable"):
        model.gradient_checkpointing_enable()

    peft_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    )

    model = get_peft_model(model, peft_config)
    model.print_trainable_parameters()

    training_args = build_training_args()
    data_collator = SupervisedDataCollator(tokenizer)

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        data_collator=data_collator,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
    )

    trainer.train()
    trainer.save_model(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)
    save_json(trainer.state.log_history, os.path.join(OUTPUT_DIR, "log_history.json"))

    print(f"Saved fine-tuned LoRA model to: {OUTPUT_DIR}")


def run_rq1_retrieval_eval():
    cleanup_memory()

    test_examples = load_examples(os.path.join(DATA_DIR, "test.json"), USE_FIELD)
    print(f"Loaded {len(test_examples)} test examples (non-empty answers only)")

    sweep_summary = threshold_sweep(test_examples, thresholds=THRESHOLD_SWEEP)
    save_json(sweep_summary, os.path.join(OUTPUT_DIR, "threshold_sweep_summary.json"))

    sparql_list = [x["sparql"] for x in test_examples]

    qwen_encoder = QwenSPARQLEncoder(OUTPUT_DIR, device=DEVICE)

    emb = qwen_encoder.encode_sparql(
        sparql_list,
        batch_size=EMBED_BATCH_SIZE,
        max_length=MAX_SEQ_LENGTH,
    )

    sim_matrix = cosine_sim_matrix(emb, emb)

    all_threshold_results = {}
    ranked_for_main = None
    jaccard_for_main = None

    for t in THRESHOLD_SWEEP:
        gt_t, jmat_t = build_ground_truth_by_answer_jaccard(
            test_examples,
            threshold=t,
            exclude_self=True,
        )

        results_t, ranked_t = evaluate_ranking(
            sim_matrix=sim_matrix,
            gt=gt_t,
            jaccard_matrix=jmat_t,
            ks=TOPK,
        )

        print_results(f"Qwen fine-tuned SPARQL encoder (threshold={t})", results_t)
        all_threshold_results[str(t)] = results_t

        if abs(t - ANSWER_JACCARD_THRESHOLD) < 1e-9:
            ranked_for_main = ranked_t
            jaccard_for_main = jmat_t

    save_json(
        all_threshold_results,
        os.path.join(OUTPUT_DIR, "rq1_retrieval_metrics_by_threshold.json"),
    )

    if ranked_for_main is not None and jaccard_for_main is not None:
        print_retrieval_examples(
            title=f"Qwen fine-tuned SPARQL encoder (threshold={ANSWER_JACCARD_THRESHOLD})",
            ranked=ranked_for_main,
            examples=test_examples,
            jaccard_matrix=jaccard_for_main,
            num_examples=5,
        )


def main():
    set_seed(SEED)

    print("Step 1/2: Fine-tuning Qwen on SPARQL -> natural text...")
    finetune_qwen_sparql2text()

    print("\nStep 2/2: Evaluating RQ1 with fine-tuned Qwen as SPARQL encoder...")
    run_rq1_retrieval_eval()


if __name__ == "__main__":
    main()

/upb/users/g/ganeshk/profiles/unix/cs/.conda/envs/sparql-emb/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Step 1/2: Fine-tuning Qwen on SPARQL -> natural text...

=== Tokenization Stats (train) ===
Total examples         : 10551
Min length             : 60
Max length             : 222
Mean length            : 91.4
Truncated (>768) : 0 (0.0%)

=== Tokenization Stats (validation) ===
Total examples         : 1526
Min length             : 63
Max length             : 176
Mean length            : 90.9
Truncated (>768) : 0 (0.0%)


Loading weights: 100%|██████████| 339/339 [00:05<00:00, 65.45it/s, Materializing param=model.norm.weight]                               


trainable params: 10,092,544 || all params: 7,625,709,056 || trainable%: 0.1323


Step,Training Loss,Validation Loss
100,2.767047,2.794336
200,2.649905,2.679739
300,2.588696,2.613150
400,2.515724,2.553948
500,2.657716,2.497456
600,2.462236,2.460050
700,2.442894,2.413863
800,2.271452,2.381093
900,2.303106,2.333229
1000,2.098122,2.327186


...


=== Qwen fine-tuned SPARQL encoder (threshold=0.6) ===
Recall@1: 0.2603
Recall@5: 0.4144
Recall@10: 0.4966
MRR@10: 0.3251
nDCG@1: 0.2639
BestJaccard@1: 0.2570
MeanJaccard@1: 0.2570
nDCG@5: 0.2285
BestJaccard@5: 0.4073
MeanJaccard@5: 0.1063
nDCG@10: 0.2256
BestJaccard@10: 0.4905
MeanJaccard@10: 0.0772
NumAnchorsWithPositives: 584


Building answer-Jaccard matrix @ threshold=0.7: 100%|██████████| 3052/3052 [01:57<00:00, 25.93it/s]



=== Qwen fine-tuned SPARQL encoder (threshold=0.7) ===
Recall@1: 0.2544
Recall@5: 0.4077
Recall@10: 0.4913
MRR@10: 0.3186
nDCG@1: 0.2581
BestJaccard@1: 0.2552
MeanJaccard@1: 0.2552
nDCG@5: 0.2229
BestJaccard@5: 0.4058
MeanJaccard@5: 0.1061
nDCG@10: 0.2200
BestJaccard@10: 0.4905
MeanJaccard@10: 0.0775
NumAnchorsWithPositives: 574


Building answer-Jaccard matrix @ threshold=0.8: 100%|██████████| 3052/3052 [01:58<00:00, 25.69it/s]



=== Qwen fine-tuned SPARQL encoder (threshold=0.8) ===
Recall@1: 0.2491
Recall@5: 0.4057
Recall@10: 0.4911
MRR@10: 0.3147
nDCG@1: 0.2529
BestJaccard@1: 0.2528
MeanJaccard@1: 0.2528
nDCG@5: 0.2177
BestJaccard@5: 0.4066
MeanJaccard@5: 0.1067
nDCG@10: 0.2147
BestJaccard@10: 0.4931
MeanJaccard@10: 0.0784
NumAnchorsWithPositives: 562

===== Retrieval preview: Qwen fine-tuned SPARQL encoder (threshold=0.8) =====

ANCHOR #0 | id=2
SPARQL:
SELECT ?answer WHERE { wd:Q16538 wdt:P725 ?answer . ?answer wdt:P106 wd:Q177220}
GOLD QUESTION:
Which female actress is the voice over on South Park and is employed as a singer?
GOLD ANSWER: ['Trey Parker', 'John Lydon', 'Ozzy Osbourne', 'Isaac Hayes', 'Jay Leno', 'Mona Marshall', 'Tim Armstrong', 'Tomorô Taguchi']

Top-5 retrieved SPARQL queries:

[Rank 1] corpus_idx=233
Question: Who is the pianist artist of The Rise and Fall of Ziggy Stardust and The Spiders from Mars?
SPARQL: SELECT ?answer WHERE { wd:Q682846 wdt:P175 ?answer . ?answer wdt:P106 wd:Q4867

In [1]:
# ============================================================
# CELL 2: Retrieval Evaluation using the fine-tuned Qwen encoder
# Metrics aligned with your earlier baseline evaluation
# ============================================================

import os
import gc
import json
from typing import Any, Dict, List, Tuple, Optional

import numpy as np
import torch
from peft import PeftConfig, PeftModel
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# ============================================================
# Config
# ============================================================

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
DATA_DIR = "dataset"

USE_FIELD = "sparql_wikidata"   # change if needed
SPARQL_VARIANT = USE_FIELD.replace("sparql_", "")
OUTPUT_DIR = f"./qwen_sparql2text_encoder_rq1_{SPARQL_VARIANT}"

MAX_SEQ_LENGTH = 768
EMBED_BATCH_SIZE = 8

ANSWER_JACCARD_THRESHOLD = 0.8
THRESHOLD_SWEEP = [0.5, 0.6, 0.7, 0.8]
TOPKS = [1, 5, 10]

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ============================================================
# Utils
# ============================================================

def cleanup_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def load_json(path: str):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def save_json(obj: Any, path: str):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def l2_normalize(x: torch.Tensor, eps: float = 1e-12) -> torch.Tensor:
    return x / x.norm(dim=-1, keepdim=True).clamp(min=eps)

def cosine_sim_matrix(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    a_norm = a / (np.linalg.norm(a, axis=1, keepdims=True) + 1e-12)
    b_norm = b / (np.linalg.norm(b, axis=1, keepdims=True) + 1e-12)
    return a_norm @ b_norm.T

# ============================================================
# Prompting / data loading
# ============================================================

def format_sparql_to_text_prompt(sparql: str) -> str:
    return (
        "You are an expert in knowledge graphs.\n\n"
        "### Instruction\n"
        "Convert the SPARQL query into a clear natural language question.\n\n"
        f"### SPARQL Query\n{sparql}\n\n"
        "### Natural Text\n"
    )

def safe_list(x: Any) -> List[str]:
    if x is None:
        return []
    if isinstance(x, list):
        return [str(v).strip() for v in x if str(v).strip()]
    s = str(x).strip()
    return [s] if s else []

def normalize_answer_item(x: str) -> str:
    return " ".join(str(x).strip().split())

def normalize_answer_list(answer: Any) -> List[str]:
    vals = safe_list(answer)
    return [normalize_answer_item(v) for v in vals if normalize_answer_item(v)]

def load_examples(path: str, use_field: str) -> List[Dict[str, Any]]:
    raw = load_json(path)
    items = []

    for i, ex in enumerate(raw):
        sparql = ex.get(use_field)
        question = ex.get("question")
        answers = normalize_answer_list(ex.get("answer"))

        if sparql is None or not str(sparql).strip():
            continue
        if question is None or not str(question).strip():
            continue
        if len(answers) == 0:
            continue

        items.append(
            {
                "id": ex.get("id", i),
                "sparql": str(sparql).strip(),
                "question": str(question).strip(),
                "answer": answers,
            }
        )

    return items

# ============================================================
# Encoder
# ============================================================

class QwenSPARQLEncoder:
    def __init__(self, lora_path: str, device: str = DEVICE):
        self.device = device

        peft_cfg = PeftConfig.from_pretrained(lora_path)
        self.base_model_name = peft_cfg.base_model_name_or_path

        try:
            self.tokenizer = AutoTokenizer.from_pretrained(lora_path)
        except Exception:
            self.tokenizer = AutoTokenizer.from_pretrained(self.base_model_name)

        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        if device == "cuda":
            bnb_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_use_double_quant=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch.float16,
            )
            base_model = AutoModelForCausalLM.from_pretrained(
                self.base_model_name,
                quantization_config=bnb_config,
                device_map="auto",
            )
        else:
            base_model = AutoModelForCausalLM.from_pretrained(
                self.base_model_name,
                torch_dtype=torch.float32,
            ).to(device)

        self.model = PeftModel.from_pretrained(base_model, lora_path, is_trainable=False)
        self.model.eval()

    def build_encoder_prompt(self, sparql: str) -> str:
        return format_sparql_to_text_prompt(sparql)

    @torch.no_grad()
    def encode_sparql(
        self,
        sparql_list: List[str],
        batch_size: int = 8,
        max_length: int = 768,
    ) -> np.ndarray:
        all_embeddings = []
        model_device = next(self.model.parameters()).device

        for start in tqdm(range(0, len(sparql_list), batch_size), desc="Encoding SPARQL"):
            batch = sparql_list[start:start + batch_size]
            prompts = [self.build_encoder_prompt(s) for s in batch]

            toks = self.tokenizer(
                prompts,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=max_length,
            )
            toks = {k: v.to(model_device) for k, v in toks.items()}

            outputs = self.model(
                **toks,
                output_hidden_states=True,
                return_dict=True,
            )
            last_hidden = outputs.hidden_states[-1]

            attn = toks["attention_mask"].unsqueeze(-1).float()
            pooled = (last_hidden * attn).sum(dim=1) / attn.sum(dim=1).clamp(min=1e-9)
            pooled = l2_normalize(pooled)

            all_embeddings.append(pooled.cpu())

        return torch.cat(all_embeddings, dim=0).numpy()

# ============================================================
# Ground truth + Jaccard relevance
# ============================================================

def answer_jaccard(a: List[str], b: List[str]) -> float:
    sa, sb = set(a), set(b)
    if not sa and not sb:
        return 0.0
    if len(sa) == 0 or len(sb) == 0:
        return 0.0
    return len(sa.intersection(sb)) / len(sa.union(sb))

def build_jaccard_matrix(
    examples: List[Dict[str, Any]],
    exclude_self: bool = True,
) -> np.ndarray:
    n = len(examples)
    jaccard_matrix = np.zeros((n, n), dtype=np.float32)

    for i in tqdm(range(n), desc="Building answer-Jaccard matrix"):
        ai = examples[i]["answer"]
        for j in range(n):
            if exclude_self and i == j:
                continue
            aj = examples[j]["answer"]
            jaccard_matrix[i, j] = answer_jaccard(ai, aj)

    return jaccard_matrix

def build_gt_from_jaccard_matrix(
    jaccard_matrix: np.ndarray,
    threshold: float,
    exclude_self: bool = True,
) -> Dict[int, List[int]]:
    n = jaccard_matrix.shape[0]
    gt: Dict[int, List[int]] = {}

    for i in range(n):
        positives = [j for j in range(n) if jaccard_matrix[i, j] >= threshold]
        if exclude_self:
            positives = [j for j in positives if j != i]
        gt[i] = positives

    return gt

def threshold_sweep_summary(jaccard_matrix: np.ndarray, thresholds: List[float]) -> Dict[str, Any]:
    n = jaccard_matrix.shape[0]

    print("\n=== Jaccard Threshold Sensitivity ===")
    print(f"{'Threshold':>10} | {'Valid Anchors':>14} | {'Coverage %':>10}")
    print("-" * 42)

    summary = {}
    for t in thresholds:
        gt_t = build_gt_from_jaccard_matrix(jaccard_matrix, threshold=t, exclude_self=True)
        valid = sum(1 for i in range(n) if len(gt_t.get(i, [])) > 0)
        pct = 100.0 * valid / max(n, 1)
        print(f"{t:>10.1f} | {valid:>14} | {pct:>9.1f}%")
        summary[str(t)] = {
            "valid_anchors": valid,
            "coverage_percent": pct,
        }

    return summary

# ============================================================
# Ranking helpers
# ============================================================

def mask_self_matches(sim_matrix: np.ndarray) -> np.ndarray:
    sim = sim_matrix.copy()
    n, m = sim.shape
    if n == m:
        diag = np.arange(n)
        sim[diag, diag] = -1e9
    return sim

def get_ranked_indices(sim_matrix: np.ndarray) -> np.ndarray:
    return np.argsort(-sim_matrix, axis=1)

# ============================================================
# Metrics aligned with baseline code
# ============================================================

def dcg(rels: List[float]) -> float:
    return float(sum((r / np.log2(i + 2)) for i, r in enumerate(rels)))

def avg_jaccard_at_k(
    ranked_indices: np.ndarray,
    jaccard_matrix: np.ndarray,
    gt: Dict[int, List[int]],
    k: int
) -> float:
    total = 0.0
    count = 0

    for i in range(ranked_indices.shape[0]):
        if len(gt.get(i, [])) == 0:
            continue
        topk = ranked_indices[i, :k]
        if len(topk) == 0:
            continue
        total += float(np.mean(jaccard_matrix[i, topk]))
        count += 1

    return total / max(count, 1)

def avg_ndcg_at_k(
    ranked_indices: np.ndarray,
    jaccard_matrix: np.ndarray,
    gt: Dict[int, List[int]],
    k: int
) -> float:
    total = 0.0
    count = 0
    n = ranked_indices.shape[0]

    for i in range(n):
        if len(gt.get(i, [])) == 0:
            continue

        topk = ranked_indices[i, :k]
        if len(topk) == 0:
            continue

        rels = [float(jaccard_matrix[i, j]) for j in topk]
        dcg_val = dcg(rels)

        ideal_rels = sorted(
            [float(jaccard_matrix[i, j]) for j in range(n) if j != i],
            reverse=True
        )[:k]
        idcg = dcg(ideal_rels)

        ndcg = (dcg_val / idcg) if idcg > 0 else 0.0
        total += ndcg
        count += 1

    return total / max(count, 1)

def avg_prf_at_k_binary(
    ranked_indices: np.ndarray,
    gt: Dict[int, List[int]],
    k: int
) -> Tuple[float, float, float]:
    p_sum = r_sum = f1_sum = 0.0
    total = 0

    for i in range(ranked_indices.shape[0]):
        pos_set = set(gt.get(i, []))
        if not pos_set:
            continue

        topk_set = set(ranked_indices[i, :k])
        inter = len(topk_set & pos_set)

        p = inter / k if k > 0 else 0.0
        r = inter / len(pos_set) if len(pos_set) > 0 else 0.0
        f1 = (2 * p * r / (p + r)) if (p + r) > 0 else 0.0

        p_sum += p
        r_sum += r
        f1_sum += f1
        total += 1

    if total == 0:
        return 0.0, 0.0, 0.0

    return p_sum / total, r_sum / total, f1_sum / total

def evaluate_ranking_with_baseline_metrics(
    sim_matrix: np.ndarray,
    jaccard_matrix: np.ndarray,
    threshold: float,
    ks: List[int],
):
    sim_matrix = mask_self_matches(sim_matrix)
    ranked = get_ranked_indices(sim_matrix)

    gt = build_gt_from_jaccard_matrix(
        jaccard_matrix=jaccard_matrix,
        threshold=threshold,
        exclude_self=True,
    )

    results = {
        "graded": {},
        "binary": {},
        "num_anchors_with_positives": int(sum(1 for i in range(ranked.shape[0]) if len(gt.get(i, [])) > 0)),
    }

    for k in ks:
        results["graded"][f"@{k}"] = {
            "avgJacc": avg_jaccard_at_k(ranked, jaccard_matrix, gt, k),
            "nDCG": avg_ndcg_at_k(ranked, jaccard_matrix, gt, k),
        }

    for k in ks:
        p, r, f1 = avg_prf_at_k_binary(ranked, gt, k)
        results["binary"][f"@{k}"] = {
            "P": p,
            "R": r,
            "F1": f1,
        }

    return results, ranked, gt

def print_results(title: str, results: Dict[str, Any], threshold: float):
    print(f"\n=== {title} ===")
    print(f"Anchors with >= threshold positive: {results['num_anchors_with_positives']}")

    print("\nGraded relevance (Jaccard):")
    for k in TOPKS:
        row = results["graded"][f"@{k}"]
        print(f"@{k:<2} | avgJacc: {row['avgJacc']:.4f} | nDCG: {row['nDCG']:.4f}")

    print(f"\nBinary relevance (Jaccard >= {threshold}):")
    for k in TOPKS:
        row = results["binary"][f"@{k}"]
        print(f"@{k:<2} | P: {row['P']:.4f} | R: {row['R']:.4f} | F1: {row['F1']:.4f}")

def print_retrieval_examples(
    title: str,
    ranked: np.ndarray,
    examples: List[Dict[str, Any]],
    jaccard_matrix: np.ndarray,
    num_examples: int = 5,
):
    print(f"\n===== Retrieval preview: {title} =====")
    shown = 0

    for i in range(len(examples)):
        topk = ranked[i, :5]
        if topk.size == 0:
            continue

        print("\n" + "=" * 80)
        print(f"ANCHOR #{i} | id={examples[i]['id']}")
        print(f"SPARQL:\n{examples[i]['sparql']}")
        print(f"GOLD QUESTION:\n{examples[i]['question']}")
        print(f"GOLD ANSWER: {examples[i]['answer']}")

        print("\nTop-5 retrieved SPARQL queries:")
        for rank, j in enumerate(topk, start=1):
            print(f"\n[Rank {rank}] corpus_idx={j}")
            print(f"Question: {examples[j]['question']}")
            print(f"SPARQL: {examples[j]['sparql'][:200]}...")
            print(f"Answer: {examples[j]['answer']}")
            print(f"Answer-Jaccard(anchor, retrieved) = {jaccard_matrix[i, j]:.4f}")

        print("\nSummary:")
        print(f"Top-1 Jaccard: {jaccard_matrix[i, topk[0]]:.4f}")
        print(f"Best Jaccard in Top-3: {np.max(jaccard_matrix[i, topk[:3]]):.4f}")
        print(f"Best Jaccard in Top-5: {np.max(jaccard_matrix[i, topk[:5]]):.4f}")

        shown += 1
        if shown >= num_examples:
            break

# ============================================================
# Evaluation
# ============================================================

def run_rq1_retrieval_eval():
    cleanup_memory()

    test_examples = load_examples(os.path.join(DATA_DIR, "test.json"), USE_FIELD)
    print(f"Loaded {len(test_examples)} test examples (non-empty answers only)")

    sparql_list = [x["sparql"] for x in test_examples]

    # Build Jaccard matrix once
    jaccard_matrix = build_jaccard_matrix(test_examples, exclude_self=True)

    # Threshold sensitivity summary
    sweep_summary = threshold_sweep_summary(jaccard_matrix, thresholds=THRESHOLD_SWEEP)
    save_json(sweep_summary, os.path.join(OUTPUT_DIR, "threshold_sweep_summary.json"))

    # Encode once
    qwen_encoder = QwenSPARQLEncoder(OUTPUT_DIR, device=DEVICE)

    emb = qwen_encoder.encode_sparql(
        sparql_list,
        batch_size=EMBED_BATCH_SIZE,
        max_length=MAX_SEQ_LENGTH,
    )

    sim_matrix = cosine_sim_matrix(emb, emb)

    all_threshold_results = {}
    ranked_for_main = None

    for t in THRESHOLD_SWEEP:
        results_t, ranked_t, gt_t = evaluate_ranking_with_baseline_metrics(
            sim_matrix=sim_matrix,
            jaccard_matrix=jaccard_matrix,
            threshold=t,
            ks=TOPKS,
        )

        print_results(
            title=f"Qwen fine-tuned SPARQL encoder (threshold={t})",
            results=results_t,
            threshold=t,
        )

        all_threshold_results[str(t)] = results_t

        if abs(t - ANSWER_JACCARD_THRESHOLD) < 1e-9:
            ranked_for_main = ranked_t

    save_json(
        all_threshold_results,
        os.path.join(OUTPUT_DIR, "rq1_retrieval_metrics_by_threshold.json"),
    )

    if ranked_for_main is not None:
        print_retrieval_examples(
            title=f"Qwen fine-tuned SPARQL encoder (threshold={ANSWER_JACCARD_THRESHOLD})",
            ranked=ranked_for_main,
            examples=test_examples,
            jaccard_matrix=jaccard_matrix,
            num_examples=5,
        )

# Run evaluation
run_rq1_retrieval_eval()

/upb/users/g/ganeshk/profiles/unix/cs/.conda/envs/sparql-emb/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 3052 test examples (non-empty answers only)


Building answer-Jaccard matrix: 100%|██████████| 3052/3052 [02:03<00:00, 24.77it/s]



=== Jaccard Threshold Sensitivity ===
 Threshold |  Valid Anchors | Coverage %
------------------------------------------
       0.5 |            677 |      22.2%
       0.6 |            584 |      19.1%
       0.7 |            574 |      18.8%
       0.8 |            562 |      18.4%


Encoding SPARQL: 100%|██████████| 382/382 [01:54<00:00,  3.34it/s]



=== Qwen fine-tuned SPARQL encoder (threshold=0.5) ===
Anchors with >= threshold positive: 677

Graded relevance (Jaccard):
@1  | avgJacc: 0.2310 | nDCG: 0.2458
@5  | avgJacc: 0.0961 | nDCG: 0.2141
@10 | avgJacc: 0.0697 | nDCG: 0.2117

Binary relevance (Jaccard >= 0.5):
@1  | P: 0.2334 | R: 0.1184 | F1: 0.1320
@5  | P: 0.0957 | R: 0.1794 | F1: 0.0891
@10 | P: 0.0690 | R: 0.2047 | F1: 0.0720

=== Qwen fine-tuned SPARQL encoder (threshold=0.6) ===
Anchors with >= threshold positive: 584

Graded relevance (Jaccard):
@1  | avgJacc: 0.2570 | nDCG: 0.2639
@5  | avgJacc: 0.1063 | nDCG: 0.2285
@10 | avgJacc: 0.0772 | nDCG: 0.2256

Binary relevance (Jaccard >= 0.6):
@1  | P: 0.2603 | R: 0.1378 | F1: 0.1511
@5  | P: 0.1041 | R: 0.2000 | F1: 0.0960
@10 | P: 0.0755 | R: 0.2266 | F1: 0.0775

=== Qwen fine-tuned SPARQL encoder (threshold=0.7) ===
Anchors with >= threshold positive: 574

Graded relevance (Jaccard):
@1  | avgJacc: 0.2552 | nDCG: 0.2581
@5  | avgJacc: 0.1061 | nDCG: 0.2229
@10 | avgJa